<a href="https://colab.research.google.com/github/jessicalewinter/deep-learning-specialization/blob/main/course-notebooks/deep-architectures/Lab02_Convolucao_por_dentro_aluno.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 2 · Convolução por dentro

**Arquiteturas Profundas e Representação de Dados** · Especialização em Deep Learning · CIn.AI / CIn-UFPE

Neste lab você abre a CNN do Lab 1 e confere, com as próprias mãos, cada número da aula 2.

- **Parte A (em aula, guiada):** convolução à mão, a fórmula do tamanho da saída, pesos e contas, um filtro aprendido por gradiente e a velocidade de três implementações.
- **Parte B (em casa):** campo receptivo teórico e efetivo, medido no cachorro.
- **Parte C (em casa):** pooling, stride ou dilatação? Três variantes da CNN do Lab 1.
- **Desafio extra (opcional, não conta para a nota):** quantas previsões mudam quando a imagem anda 1 pixel.

**Antes de começar**

1. *Arquivo → Salvar uma cópia no Drive*.
2. *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4* (a parte A também roda bem na CPU).
3. Rode as células na ordem. As células **Verificação** dizem `OK` ou `ERRO`.
4. No fim, a última célula gera o seu **código de conclusão** para o formulário de entrega.

## Parte A · em aula

### Preparação · configuração, dados e a CNN do Lab 1

In [1]:
# Célula 1 · Configuração
import os, time, json, hashlib, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import matplotlib.pyplot as plt

SEED = 2026
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
# Na CPU há menos imagens; mini-batches menores e mais épocas compensam o número menor de passos.
EPOCAS, BS = (5, 128) if DEVICE == 'cuda' else (8, 64)
print('GPU:', torch.cuda.get_device_name(0)) if DEVICE == 'cuda' else print('Sem GPU: as partes B e C usam 10 mil imagens na CPU.')

CHECKS = {}
def check(chave, condicao, ok, erro):
    """Registra uma verificação e mostra OK ou ERRO."""
    CHECKS[chave] = bool(condicao)
    print(('OK    ' + ok) if condicao else ('ERRO  ' + erro))

GPU: Tesla T4


In [2]:
# Célula 2 · CIFAR-10 (como no Lab 1)
ESPELHO_URL = None   # "https://drive.google.com/file/d/1fBXTy06cRH4M4fugrFSQOwqvhjOfAqDS/view?usp=sharing"

t0 = time.time()
if ESPELHO_URL:
    if 'drive.google.com' in ESPELHO_URL:
        import gdown                      # já vem no Colab; baixa arquivos grandes do Google Drive
        gdown.download(ESPELHO_URL, 'cifar10.npz', quiet=False, fuzzy=True)
    else:
        import urllib.request
        urllib.request.urlretrieve(ESPELHO_URL, 'cifar10.npz')
    d = np.load('cifar10.npz')
    tr_x, tr_y, te_x, te_y = d['x_train'], d['y_train'], d['x_test'], d['y_test']
else:
    tr = torchvision.datasets.CIFAR10(root='dados', train=True, download=True)
    te = torchvision.datasets.CIFAR10(root='dados', train=False, download=True)
    tr_x, tr_y, te_x, te_y = tr.data, np.array(tr.targets), te.data, np.array(te.targets)

N_TREINO, N_TESTE = (len(tr_x), len(te_x)) if DEVICE == 'cuda' else (10_000, 2_000)
def para_tensor(x, y, n):
    return torch.from_numpy(x[:n]).permute(0, 3, 1, 2).float().div(255), torch.from_numpy(np.asarray(y[:n])).long()
x_treino, y_treino = para_tensor(tr_x, tr_y, N_TREINO)
x_teste, y_teste = para_tensor(te_x, te_y, N_TESTE)
MEDIA = x_treino.mean(dim=(0, 2, 3), keepdim=True); DESVIO = x_treino.std(dim=(0, 2, 3), keepdim=True)
x_treino = ((x_treino - MEDIA) / DESVIO).to(DEVICE); x_teste = ((x_teste - MEDIA) / DESVIO).to(DEVICE)
y_treino, y_teste = y_treino.to(DEVICE), y_teste.to(DEVICE)

IDX_CACHORRO = int((y_teste == 5).nonzero()[1])   # o mesmo cachorro do Lab 1 e dos slides
def para_imagem(x):
    """Desfaz a normalização e devolve uma imagem H × W × 3 em [0, 1]."""
    return (x.detach().cpu() * DESVIO[0] + MEDIA[0]).clamp(0, 1).permute(1, 2, 0).numpy()
print(f'Treino: {tuple(x_treino.shape)} · Teste: {tuple(x_teste.shape)} · {time.time() - t0:.0f}s')
check('dados', x_treino.shape[1:] == (3, 32, 32) and len(x_treino) == N_TREINO, 'dados carregados', 'formato inesperado: rode a célula 2 de novo')

100%|██████████| 170M/170M [26:27<00:00, 107kB/s]


Treino: (50000, 3, 32, 32) · Teste: (10000, 3, 32, 32) · 1595s
OK    dados carregados


In [3]:
# Célula 3 · A CNN do Lab 1 e a função de treino (prontas)
def CNN():
    return nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10))

def avaliar(model, x, y, bs=1000):
    model.eval(); acertos = 0
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
        for i in range(0, len(x), bs):
            acertos += (model(x[i:i + bs]).argmax(1) == y[i:i + bs]).sum().item()
    return acertos / len(x)

def treinar(model, epocas=EPOCAS, bs=BS, lr=1e-3, nome='modelo'):
    """Treina com Adam no CIFAR-10 (como no Lab 1). Devolve o modelo, a acurácia de teste e o tempo."""
    torch.manual_seed(SEED); model = model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr); scaler = torch.amp.GradScaler(enabled=USE_AMP); t0 = time.time()
    for ep in range(epocas):
        model.train(); ordem = torch.randperm(len(x_treino), device=DEVICE)
        for i in range(0, len(x_treino), bs):
            b = ordem[i:i + bs]
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
                perda = F.cross_entropy(model(x_treino[b]), y_treino[b])
            opt.zero_grad(set_to_none=True); scaler.scale(perda).backward(); scaler.step(opt); scaler.update()
    acc = avaliar(model, x_teste, y_teste); dt = time.time() - t0
    print(f'{nome}: acurácia de teste {acc:.1%} · {dt:.0f}s')
    return model, acc, dt

cnn = CNN()
print(cnn)

Sequential(
  (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (7): ReLU()
  (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (9): AdaptiveAvgPool2d(output_size=1)
  (10): Flatten(start_dim=1, end_dim=-1)
  (11): Linear(in_features=128, out_features=10, bias=True)
)


### Passo 1 · Convolução à mão (10 min)

Uma camada convolucional com vários canais, escrita com laços. A entrada tem `Cin` canais; cada um dos `Cout` filtros tem `Cin × k × k` pesos e um viés. Em cada posição, a janela da entrada (todos os canais) é multiplicada pelo filtro e somada.

In [4]:
# Célula 4 · Convolução com laços
def conv2d_lacos(x, w, b, stride=1, padding=0):
    """x: (Cin, H, W) · w: (Cout, Cin, k, k) · b: (Cout,). Devolve (Cout, H', W')."""
    Cin, H, W = x.shape
    Cout, _, k, _ = w.shape
    xp = F.pad(x, (padding, padding, padding, padding))          # zeros em volta
    Hs = (H + 2 * padding - k) // stride + 1
    Ws = (W + 2 * padding - k) // stride + 1
    y = torch.zeros(Cout, Hs, Ws)
    for co in range(Cout):
        for i in range(Hs):
            for j in range(Ws):
                # TODO: pegue a janela xp[:, i*stride : i*stride + k, j*stride : j*stride + k] (Cin × k × k),
                #       multiplique pelo filtro w[co], some tudo e some o viés b[co]. Guarde em y[co, i, j].
                raise NotImplementedError('Complete o TODO acima')
    return y

**Verificação.**

In [5]:
cachorro = x_teste[IDX_CACHORRO].cpu()
torch.manual_seed(0); _w, _b = torch.randn(8, 3, 3, 3), torch.randn(8)
_ok = all(torch.allclose(conv2d_lacos(cachorro, _w, _b, s, p), F.conv2d(cachorro[None], _w, _b, stride=s, padding=p)[0], atol=1e-4)
          for s, p in [(1, 1), (2, 0), (2, 1)])
check('conv_lacos', _ok, 'a sua convolução bate com F.conv2d (stride 1 e 2, com e sem padding)',
      'resultado diferente de F.conv2d: confira a janela e a soma')

NotImplementedError: Complete o TODO acima

### Passo 2 · O tamanho da saída (8 min)

Complete a fórmula da aula: saída = ⌊(n + 2p − d(k − 1) − 1) / s⌋ + 1.

In [ ]:
# Célula 5 · A fórmula
def tamanho_saida(n, k, p=0, s=1, d=1):
    """Tamanho da saída de uma convolução (ou pooling) em uma dimensão."""
    # TODO: implemente a fórmula (use // para a divisão inteira)
    raise NotImplementedError('Complete o TODO acima')


**Verificação.**

In [ ]:
_rng = random.Random(7); _acertos = _total = 0
for _ in range(80):
    n, k, p, s, d = _rng.randint(5, 40), _rng.choice([1, 2, 3, 5, 7]), _rng.randint(0, 3), _rng.randint(1, 3), _rng.randint(1, 3)
    if (n + 2 * p - d * (k - 1) - 1) < 0 or p > d * (k - 1): continue
    _total += 1
    _acertos += tamanho_saida(n, k, p, s, d) == nn.Conv2d(1, 1, k, s, p, d)(torch.zeros(1, 1, n, n)).shape[-1]
check('tamanho', _acertos == _total and _total >= 50, f'a fórmula bate com o PyTorch em {_acertos} de {_total} configurações',
      f'a fórmula bateu em {_acertos} de {_total}: confira o d·(k − 1) e a divisão inteira')

Agora preveja, **sem rodar a rede**, o formato (canais, altura, largura) da saída de cada camada da CNN do Lab 1 para uma imagem 3 × 32 × 32. As convoluções usam `padding=1`; os poolings, janela 2 e stride 2.

In [ ]:
# Célula 6 · Prever os formatos da CNN do Lab 1
# TODO: troque cada None por (C, H, W)
PREVISAO = {'conv1': None, 'pool1': None, 'conv2': None, 'pool2': None, 'conv3': None, 'pool3': None}

# conferir com a rede de verdade
_nomes = {0: 'conv1', 2: 'pool1', 3: 'conv2', 5: 'pool2', 6: 'conv3', 8: 'pool3'}
_x = torch.zeros(1, 3, 32, 32); REAL = {}
for i, m in enumerate(cnn):
    _x = m(_x)
    if i in _nomes: REAL[_nomes[i]] = tuple(_x.shape[1:])
for nome in REAL: print(f'{nome}: previsto {PREVISAO[nome]} · real {REAL[nome]}')
check('formatos', all(PREVISAO[n] == REAL[n] for n in REAL), 'os seis formatos previstos estão certos', 'algum formato não bate: use tamanho_saida')

### Passo 3 · Pesos e contas (8 min)

Pesos = Cout · (Cin · k · k + 1). Contas (multiplicações-acumulações) = H' · W' · Cout · Cin · k · k.

In [ ]:
# Célula 7 · Custo de uma camada convolucional
def custo_conv(cin, cout, k, h_saida, w_saida, vies=True):
    """Devolve (pesos, contas) de uma camada convolucional."""
    # TODO: as duas fórmulas acima
    raise NotImplementedError('Complete o TODO acima')

def tabela_custo(model, entrada=(3, 32, 32)):
    """Percorre a rede com uma imagem falsa e soma pesos e contas de cada camada."""
    x = torch.zeros(1, *entrada); linhas = []; tot_p = tot_c = 0
    for m in model:
        y = m(x)
        if isinstance(m, nn.Conv2d):
            p, c = custo_conv(m.in_channels, m.out_channels, m.kernel_size[0], y.shape[2], y.shape[3])
            linhas.append((f'conv {m.kernel_size[0]}×{m.kernel_size[0]}, {m.out_channels}', tuple(y.shape[1:]), p, c))
        elif isinstance(m, nn.Linear):
            p, c = m.in_features * m.out_features + m.out_features, m.in_features * m.out_features
            linhas.append((f'linear {m.in_features} → {m.out_features}', tuple(y.shape[1:]), p, c))
        else:
            p = c = 0
        tot_p += p; tot_c += c; x = y
    return linhas, tot_p, tot_c

linhas, pesos_cnn, contas_cnn = tabela_custo(cnn)
print(f'{"camada":<20}{"saída":>16}{"pesos":>12}{"contas":>14}')
for nome, forma, p, c in linhas: print(f'{nome:<20}{str(forma):>16}{p:>12,}{c:>14,}'.replace(',', '.'))
pesos_mlp = 3072 * 512 + 512 + 512 * 256 + 256 + 256 * 10 + 10
contas_mlp = 3072 * 512 + 512 * 256 + 256 * 10
print(f'\nCNN: {pesos_cnn:,} pesos · {contas_cnn:,} contas'.replace(',', '.'))
print(f'MLP: {pesos_mlp:,} pesos · {contas_mlp:,} contas'.replace(',', '.'))
print(f'A CNN tem {pesos_mlp / pesos_cnn:.0f} vezes menos pesos e faz {contas_cnn / contas_mlp:.1f} vezes mais contas.')

**Verificação.**

In [ ]:
check('custo', pesos_cnn == sum(p.numel() for p in cnn.parameters()) == 94_538 and contas_cnn == 10_323_200,
      'pesos (94.538) e contas (10.323.200) da CNN conferem', 'os totais não batem: confira custo_conv')

### Passo 4 · Aprender um filtro (12 min)

Exemplo do *Dive into Deep Learning* (seção 7.2): aplicamos um filtro de borda conhecido ao cachorro (em cinza) e pedimos ao gradiente que **descubra** esse filtro só olhando a entrada e a saída. A imagem é normalizada (média 0, desvio 1), o que deixa o problema bem mais fácil para o otimizador; a aula 5 explica por quê.

In [ ]:
# Célula 8 · Aprender os 9 pesos de um filtro
cinza = torch.tensor(para_imagem(x_teste[IDX_CACHORRO]).mean(-1))
X = ((cinza - cinza.mean()) / cinza.std())[None, None]                      # 1 × 1 × 32 × 32
K_VERDADEIRO = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])[None, None]
Y = F.conv2d(X, K_VERDADEIRO, padding=1)                                  # o mapa de bordas "observado"

torch.manual_seed(0)
K = torch.randn(1, 1, 3, 3, requires_grad=True)                           # chute inicial
opt = torch.optim.Adam([K], lr=0.1)
perdas = []
for passo in range(1500):
    # TODO: um passo de gradiente
    # 1) saída = F.conv2d(X, K, padding=1); perda = F.mse_loss(saída, Y)
    # 2) opt.zero_grad(); perda.backward(); opt.step()
    # 3) perdas.append(perda.item())
    raise NotImplementedError('Complete o TODO acima')

fig, ax = plt.subplots(1, 4, figsize=(13, 3))
ax[0].semilogy(perdas, color='#DB1E2F'); ax[0].set_title('perda'); ax[0].set_xlabel('passo')
for a, (t, m) in zip(ax[1:], [('filtro verdadeiro', K_VERDADEIRO[0, 0]), ('filtro aprendido', K.detach()[0, 0]), ('mapa de bordas (Y)', Y[0, 0])]):
    a.imshow(m, cmap='RdBu_r', vmin=-abs(m).max(), vmax=abs(m).max()); a.set_title(t); a.axis('off')
    if m.shape[0] == 3:
        for i in range(3):
            for j in range(3): a.text(j, i, f'{m[i, j]:.2f}', ha='center', va='center', fontsize=10)
plt.tight_layout(); plt.show()

**Verificação.**

In [ ]:
_erro = (K.detach() - K_VERDADEIRO).abs().max().item()
check('filtro', len(perdas) == 1500 and _erro < 0.05, f'o filtro aprendido difere do verdadeiro em no máximo {_erro:.3f}',
      f'erro máximo {_erro:.3f}: confira o passo de gradiente')

### Passo 5 · Laços × im2col × PyTorch (5 min)

O mesmo cálculo de três jeitos, numa camada com 32 filtros 3 × 3 sobre o cachorro. O `im2col` transforma cada janela numa coluna e faz **uma** multiplicação de matrizes.

In [ ]:
# Célula 9 · Velocidade
def conv2d_im2col(x, w, b, stride=1, padding=0):
    Cout, Cin, k, _ = w.shape
    colunas = F.unfold(x[None], k, padding=padding, stride=stride)[0]       # (Cin·k·k) × (H'·W')
    Hs = tamanho_saida(x.shape[1], k, padding, stride); Ws = tamanho_saida(x.shape[2], k, padding, stride)
    return (w.reshape(Cout, -1) @ colunas + b[:, None]).reshape(Cout, Hs, Ws)

torch.manual_seed(1); w32, b32 = torch.randn(32, 3, 3, 3), torch.randn(32)
def cronometrar(f, repeticoes):
    if DEVICE == 'cuda': torch.cuda.synchronize()
    t = time.perf_counter()
    for _ in range(repeticoes): y = f()
    if DEVICE == 'cuda': torch.cuda.synchronize()
    return (time.perf_counter() - t) / repeticoes, y

t_lacos, y1 = cronometrar(lambda: conv2d_lacos(cachorro, w32, b32, 1, 1), 1)
t_im2col, y2 = cronometrar(lambda: conv2d_im2col(cachorro, w32, b32, 1, 1), 50)
xg, wg, bg = cachorro[None].to(DEVICE), w32.to(DEVICE), b32.to(DEVICE)
t_torch, y3 = cronometrar(lambda: F.conv2d(xg, wg, bg, padding=1), 200)
RAZAO = t_lacos / t_torch
print(f'laços: {t_lacos * 1e3:.1f} ms · im2col: {t_im2col * 1e3:.3f} ms · PyTorch ({DEVICE}): {t_torch * 1e3:.3f} ms')
print(f'O PyTorch foi {RAZAO:,.0f} vezes mais rápido que os laços.'.replace(',', '.'))
print(f'Para o formulário da turma: {RAZAO:.0f} ({DEVICE})')
check('velocidade', torch.allclose(y1, y2, atol=1e-4) and torch.allclose(y1, y3[0].cpu(), atol=1e-3), 'as três versões dão o mesmo resultado',
      'as versões não batem: confira o passo 1')

## Parte B · em casa: campo receptivo teórico × efetivo

**Teórico:** a região da entrada que *pode* afetar um neurônio. Cada camada soma (k − 1) · d vezes o produto dos strides das camadas anteriores:
r = 1 + Σ (kₗ − 1) · dₗ · (s₁ · … · sₗ₋₁).

**Efetivo:** quanto cada pixel *de fato* afeta o neurônio, medido pelo gradiente (Luo et al., NeurIPS 2016).

In [ ]:
# Célula 10 · A fórmula do campo receptivo
def campo_receptivo(camadas):
    """camadas: lista de (k, s, d). Devolve a lista com o campo receptivo depois de cada camada."""
    # TODO: comece com r = 1 e salto = 1; para cada camada, r += (k − 1)·d·salto e depois salto *= s
    raise NotImplementedError('Complete o TODO acima')

LAB1 = [(3, 1, 1), (2, 2, 1), (3, 1, 1), (2, 2, 1), (3, 1, 1), (2, 2, 1)]   # conv, pool, conv, pool, conv, pool
print('CNN do Lab 1:', campo_receptivo(LAB1))
print('três 3×3:', campo_receptivo([(3, 1, 1)] * 3), '· dilatada 1, 2, 4, 8:', campo_receptivo([(3, 1, 1), (3, 1, 2), (3, 1, 4), (3, 1, 8)]))

**Verificação.**

In [ ]:
check('campo', campo_receptivo(LAB1) == [3, 4, 8, 10, 18, 22] and campo_receptivo([(3, 1, 1), (3, 1, 2), (3, 1, 4), (3, 1, 8)])[-1] == 31,
      'campos 3, 4, 8, 10, 18, 22 na CNN do Lab 1 e 31 na dilatada', 'os campos não batem: confira a ordem de r e do salto')

In [ ]:
# Célula 11 · Medir o campo efetivo pelo gradiente (rede sem treino × treinada)
def campo_efetivo(rede, imagens):
    """Média de |∂ neurônio central da conv3 / ∂ pixel| sobre as imagens. Devolve um mapa 32 × 32."""
    feat = rede[:8].float().eval(); g = torch.zeros(32, 32)
    for im in imagens:
        xi = im[None].float().clone().requires_grad_(True)
        feat(xi)[0, :, 4, 4].sum().backward()                     # todos os 128 canais da posição (4, 4) do mapa 8 × 8
        g += xi.grad.abs().sum(1)[0].cpu()
    return g / len(imagens)

def desvio(g):
    g = g / g.sum(); yy, xx = torch.meshgrid(torch.arange(32.), torch.arange(32.), indexing='ij')
    cy, cx = (g * yy).sum(), (g * xx).sum()
    return torch.sqrt((g * ((yy - cy) ** 2 + (xx - cx) ** 2)).sum() / 2).item()

amostra = x_teste[:100]
torch.manual_seed(0); sem_treino = CNN().to(DEVICE)
g_antes = campo_efetivo(sem_treino, amostra)
cnn_treinada, acc_pooling, t_pooling = treinar(CNN(), nome='CNN do Lab 1')
g_depois = campo_efetivo(cnn_treinada, amostra)
linhas_nz = ((g_antes + g_depois).sum(1) > 0).nonzero(); suporte = int(linhas_nz.max() - linhas_nz.min() + 1)
d_antes, d_depois = desvio(g_antes), desvio(g_depois)

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(para_imagem(x_teste[IDX_CACHORRO])); ax[0].add_patch(plt.Rectangle((8.5, 8.5), 18, 18, fill=False, ec='#DB1E2F', lw=2.5))
ax[0].set_title('campo teórico: 18×18')
for a, g, t in [(ax[1], g_antes, f'efetivo, sem treino (desvio {d_antes:.2f} px)'), (ax[2], g_depois, f'efetivo, treinada (desvio {d_depois:.2f} px)')]:
    a.imshow(g, cmap='Reds'); a.add_patch(plt.Rectangle((8.5, 8.5), 18, 18, fill=False, ec='k', lw=1, ls='--')); a.set_title(t, fontsize=10)
for a in ax: a.axis('off')
plt.tight_layout(); plt.savefig('campo_efetivo_cachorro.png', dpi=150); plt.show()
print(f'Suporte do gradiente (média de 100 imagens): {suporte} pixels · desvio do efetivo: {d_antes:.2f} → {d_depois:.2f} px')

**Verificação.**

In [ ]:
check('efetivo', suporte == 18 and d_antes < 9 and d_depois < 9,
      'o gradiente ocupa exatamente 18 pixels, e o campo efetivo é bem menor que o teórico',
      f'suporte {suporte} (esperado 18): confira se a rede é a CNN do Lab 1')

**Responda (3 a 5 linhas):** por que o campo efetivo é bem menor que o teórico de 18 × 18? E por que, nesta rede rasa, o treino quase não mudou o tamanho dele, ao contrário do que Luo et al. viram em redes profundas?

In [ ]:
# Célula 12 · Sua resposta
RESPOSTA_B = """
escreva aqui
"""
_t = RESPOSTA_B.replace('escreva aqui', '').strip()
check('resposta_b', len(_t) >= 120, 'resposta registrada', 'escreva pelo menos 3 linhas')

## Parte C · em casa: pooling, stride ou dilatação?

Três jeitos de deixar o neurônio do fim ver mais da imagem, todos com 3 convoluções de 32, 64 e 128 filtros:

- **pooling:** a CNN do Lab 1 (convolução e max pooling, três vezes);
- **stride:** sem pooling; cada convolução usa stride 2;
- **dilatada:** um pooling só, depois convoluções com dilatação 2 e 4 (o mapa fica em 16 × 16).

In [ ]:
# Célula 13 · As três variantes
def cnn_stride():
    return nn.Sequential(nn.Conv2d(3, 32, 3, stride=2, padding=1), nn.ReLU(),
                         nn.Conv2d(32, 64, 3, stride=2, padding=1), nn.ReLU(),
                         nn.Conv2d(64, 128, 3, stride=2, padding=1), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10))
def cnn_dilatada():
    return nn.Sequential(nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
                         nn.Conv2d(32, 64, 3, padding=2, dilation=2), nn.ReLU(),
                         nn.Conv2d(64, 128, 3, padding=4, dilation=4), nn.ReLU(),
                         nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10))

# TODO: descreva as camadas de cada variante como (k, s, d), até a última convolução, para calcular o campo receptivo
CAMADAS = {'pooling': [(3, 1, 1), (2, 2, 1), (3, 1, 1), (2, 2, 1), (3, 1, 1)]}
CAMADAS['stride'] = None; CAMADAS['dilatada'] = None
for nome in CAMADAS: print(nome, '→ campo da última convolução:', campo_receptivo(CAMADAS[nome])[-1])

**Verificação.**

In [ ]:
check('variantes', [campo_receptivo(CAMADAS[n])[-1] for n in ['pooling', 'stride', 'dilatada']] == [18, 15, 28],
      'campos 18 (pooling), 15 (stride) e 28 (dilatada)', 'confira as listas (k, s, d) das variantes')

In [ ]:
# Célula 14 · Treinar e comparar (a variante pooling já foi treinada na parte B)
RESULTADOS_C = {'pooling': (acc_pooling, t_pooling)}
for nome, f in [('stride', cnn_stride), ('dilatada', cnn_dilatada)]:
    _, acc, dt = treinar(f(), nome=nome)
    RESULTADOS_C[nome] = (acc, dt)
print(f'\n{"variante":<10}{"acurácia":>10}{"pesos":>10}{"contas":>14}{"campo":>7}{"tempo":>8}')
for nome, f in [('pooling', CNN), ('stride', cnn_stride), ('dilatada', cnn_dilatada)]:
    _, p, c = tabela_custo(f())
    acc, dt = RESULTADOS_C[nome]
    print(f'{nome:<10}{acc:>10.1%}{p:>10,}{c:>14,}{campo_receptivo(CAMADAS[nome])[-1]:>7}{dt:>7.0f}s'.replace(',', '.'))
check('tabela_c', len(RESULTADOS_C) == 3, 'tabela das três variantes completa', 'rode a célula 14 até o fim')

**Responda (3 a 5 linhas):** compare as três variantes em acurácia, contas e campo receptivo. Qual você escolheria para o CIFAR-10, e por quê? O que mudaria numa imagem de 224 × 224?

In [ ]:
# Célula 15 · Sua resposta
RESPOSTA_C = """
escreva aqui
"""
_t = RESPOSTA_C.replace('escreva aqui', '').strip()
check('resposta_c', len(_t) >= 120, 'resposta registrada', 'escreva pelo menos 3 linhas')

## Desafio extra · opcional: deslocar 1 pixel

Quantas previsões mudam quando todas as imagens de teste andam 1 pixel para a direita? Comparamos a CNN do Lab 1 com uma versão em que cada max pooling é seguido de um **borrão** antes de subamostrar (Zhang, *Making Convolutional Networks Shift-Invariant Again*, ICML 2019). Não conta para a nota.

In [ ]:
# Célula 16 · Consistência a deslocamentos
class BlurPool(nn.Module):
    """Borra com [1, 2, 1] ⊗ [1, 2, 1] e subamostra com stride 2, canal a canal."""
    def __init__(self, canais):
        super().__init__()
        f = torch.tensor([1., 2., 1.]); f = (f[:, None] * f[None, :]) / 16
        self.register_buffer('f', f[None, None].repeat(canais, 1, 1, 1)); self.c = canais
    def forward(self, x):
        return F.conv2d(F.pad(x, (1, 1, 1, 1), mode='reflect'), self.f.to(x.dtype), stride=2, groups=self.c)

def cnn_borrada():
    bloco = lambda ci, co: [nn.Conv2d(ci, co, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2, stride=1), BlurPool(co)]
    return nn.Sequential(*bloco(3, 32), *bloco(32, 64), *bloco(64, 128), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(128, 10))

def consistencia(model, desl=1):
    model.eval()
    with torch.no_grad(), torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=USE_AMP):
        a = torch.cat([model(x_teste[i:i + 1000]).argmax(1) for i in range(0, len(x_teste), 1000)])
        xs = torch.roll(x_teste, shifts=desl, dims=3)
        b = torch.cat([model(xs[i:i + 1000]).argmax(1) for i in range(0, len(xs), 1000)])
    return (a == b).float().mean().item()

borrada, acc_borrada, _ = treinar(cnn_borrada(), nome='CNN com borrão')
print(f'Previsões iguais após deslocar 1 pixel · CNN do Lab 1: {consistencia(cnn_treinada):.1%} · com borrão: {consistencia(borrada):.1%}')
print(f'Acurácia · CNN do Lab 1: {acc_pooling:.1%} · com borrão: {acc_borrada:.1%}')